# RAG with LangChain + `langchain_openai` —

This notebook demonstrates a complete **Retrieval-Augmented Generation (RAG)** workflow using current LangChain-style APIs.

### What is covered
- PDF document loading
- Text chunking
- OpenAI embeddings using `langchain_openai`
- Chroma vector database
- Retriever `.invoke()` usage
- Final RAG answers using `ChatOpenAI`
- MultiQueryRetriever
- Contextual Compression Retriever
- Optional Wikipedia RAG without `WikipediaLoader`
- A second banking/Basel RAG example
- Constitution of India RAG example
- Simple RAG validation with an LLM-as-a-judge

> The code uses `.env` for API keys. No API key should be hard-coded in the notebook.

## 0. Environment setup

Create a file named **`.env`** in the same folder as this notebook.

Example:

```text
OPENAI_API_KEY=your_openai_api_key_here
OPENAI_MODEL=gpt-4.1-mini
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
```

`OPENAI_MODEL` and `OPENAI_EMBEDDING_MODEL` are optional because the notebook provides defaults.

In [ ]:
%pip install -qU \
    langchain \
    langchain-classic \
    langchain-community \
    langchain-openai \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    python-dotenv \
    requests \
    pandas

> If this is the first time you installed or upgraded these packages, restart the Jupyter kernel once before continuing.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

#if not os.getenv("OPENAI_API_KEY"):
#    raise ValueError(
#        "OPENAI_API_KEY was not found. "
#        "Create a .env file in the same folder as this notebook."
#    )
os.environ['OPENAI_API_KEY'] = "your api key"

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv(
    "OPENAI_EMBEDDING_MODEL",
    "text-embedding-3-small"
)

print("Chat model:", CHAT_MODEL)
print("Embedding model:", EMBEDDING_MODEL)
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "langchain",
    "langchain-classic",
    "langchain-community",
    "langchain-openai",
    "langchain-chroma",
    "langchain-text-splitters",
    "chromadb",
]

for package in packages:
    try:
        print(f"{package:28s} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:28s} NOT INSTALLED")

## 1. Common imports and helper functions

Important modern API changes used in this notebook:

- `retriever.invoke(question)` instead of `get_relevant_documents(...)`
- `from langchain_chroma import Chroma`
- `from langchain_text_splitters import RecursiveCharacterTextSplitter`
- `ChatOpenAI` and `OpenAIEmbeddings` come from `langchain_openai`
- `create_retrieval_chain` is used instead of the older `RetrievalQA`

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import requests

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [ ]:
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL
)

print("LLM and embedding objects created.")

In [ ]:
def download_file(url, output_path):
    """Download a file and fail clearly if the HTTP request is unsuccessful."""
    output_path = Path(output_path)

    headers = {
        "User-Agent": "LangChain-RAG-Training-Demo/1.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )
    response.raise_for_status()

    output_path.write_bytes(response.content)

    print(
        f"Downloaded: {output_path} "
        f"({len(response.content):,} bytes)"
    )

    return output_path

In [ ]:
def show_documents(documents, max_chars=700):
    """Print retrieved documents in a trainer-friendly format."""
    for i, doc in enumerate(documents, start=1):
        print("=" * 90)
        print(f"DOCUMENT {i}")
        print("Metadata:", doc.metadata)
        print("-" * 90)
        print(doc.page_content[:max_chars])
        if len(doc.page_content) > max_chars:
            print("...")

In [ ]:
def create_rag_chain(retriever, model):
    """
    Connect:
        Question -> Retriever -> Relevant chunks -> ChatOpenAI -> Final answer
    """

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are a RAG assistant.
Answer the user's question using ONLY the supplied context.

Rules:
1. Do not invent information.
2. If the answer is not available in the context, say:
   "The answer is not available in the retrieved context."
3. Keep the answer clear and concise.

Context:
{context}"""
            ),
            ("human", "{input}")
        ]
    )

    document_chain = create_stuff_documents_chain(
        model,
        prompt
    )

    return create_retrieval_chain(
        retriever,
        document_chain
    )

In [ ]:
def ask_rag(rag_chain, question, show_context=True):
    """
    Execute the RAG chain and print both the generated answer
    and the retrieved source chunks.
    """
    response = rag_chain.invoke(
        {"input": question}
    )

    print("QUESTION")
    print(question)

    print("\nANSWER")
    print(response["answer"])

    if show_context:
        print("\nRETRIEVED CONTEXT")
        for i, doc in enumerate(response["context"], start=1):
            source = doc.metadata.get("source", "N/A")
            page = doc.metadata.get("page", "N/A")
            print(
                f"{i}. source={source}, page={page}"
            )

    return response

# Example 1 — Employee Agreement RAG

Pipeline:

**PDF → Pages → Chunks → OpenAI Embeddings → Chroma → Retriever → ChatOpenAI → Answer**

## Step 1 — Download and load the Employee Agreement PDF

In [ ]:
EMPLOYEE_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/Agreements/EMPLOYEE_AGREEMENT.pdf"
)

employee_pdf = download_file(
    EMPLOYEE_URL,
    "EMPLOYEE_AGREEMENT.pdf"
)

employee_loader = PyPDFLoader(
    str(employee_pdf)
)

employee_pages = employee_loader.load()

print("Pages loaded:", len(employee_pages))

In [ ]:
full_text = "\n".join(
    page.page_content
    for page in employee_pages
)

print("Pages      :", len(employee_pages))
print("Lines      :", len(full_text.splitlines()))
print("Words      :", len(full_text.split()))
print("Characters :", len(full_text))

## Step 2 — Split the document into chunks

In [ ]:
employee_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

employee_chunks = employee_splitter.split_documents(
    employee_pages
)

print("Chunks created:", len(employee_chunks))
print("\nFirst chunk:")
print(employee_chunks[0].page_content[:700])

## Step 3 — Create a sample OpenAI embedding

In [ ]:
sample_vector = embeddings.embed_query(
    "What is the sick leave policy?"
)

print("Embedding dimensions:", len(sample_vector))
print("First 10 values:", sample_vector[:10])

## Step 4 — Store the chunks in Chroma

With the current Chroma integration, specifying `persist_directory` is enough for local persistence.  
A separate `persist()` call is not required.

In [ ]:
EMP_DB_DIR = "emp_rules_db_final"

shutil.rmtree(
    EMP_DB_DIR,
    ignore_errors=True
)

emp_rules_db = Chroma.from_documents(
    documents=employee_chunks,
    embedding=embeddings,
    collection_name="employee_agreement",
    persist_directory=EMP_DB_DIR
)

print("Employee vector database created.")

## Step 5 — Retrieve relevant chunks

In [ ]:
employee_retriever = emp_rules_db.as_retriever(
    search_kwargs={"k": 4}
)

retrieved_docs = employee_retriever.invoke(
    "What is the policy for sick leaves?"
)

print("Retrieved documents:", len(retrieved_docs))
show_documents(retrieved_docs)

## Step 6 — Connect the retriever to `ChatOpenAI`

This is the actual **generation** part of RAG.

The LLM does not receive the whole PDF. It receives the chunks returned by the retriever.

In [ ]:
employee_rag = create_rag_chain(
    employee_retriever,
    llm
)

employee_response = ask_rag(
    employee_rag,
    "What is the policy for sick leaves?"
)

In [ ]:
ask_rag(
    employee_rag,
    "What is the base compensation?"
)

In [ ]:
ask_rag(
    employee_rag,
    "What does the agreement say about insurance?"
)

# Example 2 — Basel / Banking RAG

This example uses another PDF, but the same architecture:

**Banking PDF → chunks → embeddings → vector DB → retrieval → ChatOpenAI answer**

In [ ]:
BASEL_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/Banking_System_Doc/BASEL.pdf"
)

basel_pdf = download_file(
    BASEL_URL,
    "BASEL.pdf"
)

basel_pages = PyPDFLoader(
    str(basel_pdf)
).load()

print("Pages loaded:", len(basel_pages))

In [ ]:
basel_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

basel_chunks = basel_splitter.split_documents(
    basel_pages
)

print("Chunks created:", len(basel_chunks))

In [ ]:
BASEL_DB_DIR = "basel_norms_db_final"

shutil.rmtree(
    BASEL_DB_DIR,
    ignore_errors=True
)

basel_db = Chroma.from_documents(
    documents=basel_chunks,
    embedding=embeddings,
    collection_name="basel_rules",
    persist_directory=BASEL_DB_DIR
)

basel_retriever = basel_db.as_retriever(
    search_kwargs={"k": 5}
)

print("Basel vector database and retriever created.")

In [ ]:
basel_docs = basel_retriever.invoke(
    "What percentage is the minimum capital requirement?"
)

show_documents(
    basel_docs,
    max_chars=500
)

In [ ]:
basel_rag = create_rag_chain(
    basel_retriever,
    llm
)

ask_rag(
    basel_rag,
    "What percentage is the minimum capital requirement?"
)

In [ ]:
ask_rag(
    basel_rag,
    "What are PD and LGD?"
)

# Example 3 — MultiQueryRetriever + RAG

A standard retriever searches with one user query.

`MultiQueryRetriever` asks the LLM to create multiple alternative versions of the question, runs retrieval for them, and combines the unique results.

For current LangChain versions it is provided through **`langchain_classic`**.

In [ ]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=emp_rules_db.as_retriever(
        search_kwargs={"k": 3}
    ),
    llm=llm,
    include_original=True
)

print("MultiQueryRetriever created.")

In [ ]:
multi_question = (
    "What leave benefits are available "
    "when an employee is sick?"
)

multi_docs = multiquery_retriever.invoke(
    multi_question
)

print("Unique documents returned:", len(multi_docs))
show_documents(
    multi_docs,
    max_chars=500
)

## Connect MultiQueryRetriever to ChatOpenAI

In [ ]:
multiquery_rag = create_rag_chain(
    multiquery_retriever,
    llm
)

ask_rag(
    multiquery_rag,
    multi_question
)

# Example 4 — Contextual Compression Retriever + RAG

Contextual compression performs two steps:

1. Retrieve potentially relevant chunks.
2. Use the LLM to extract only the portions useful for the current question.

This can reduce irrelevant text before the final answer-generation step.

In [ ]:
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever
)
from langchain_classic.retrievers.document_compressors.chain_extract import (
    LLMChainExtractor
)

base_retriever = emp_rules_db.as_retriever(
    search_kwargs={"k": 5}
)

compressor = LLMChainExtractor.from_llm(
    llm
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

print("Contextual compression retriever created.")

In [ ]:
compression_question = (
    "What is the policy for sick leave?"
)

compressed_docs = compression_retriever.invoke(
    compression_question
)

print("Compressed documents:", len(compressed_docs))
show_documents(
    compressed_docs,
    max_chars=500
)

## Connect the compressed retriever to ChatOpenAI

In [ ]:
compression_rag = create_rag_chain(
    compression_retriever,
    llm
)

ask_rag(
    compression_rag,
    compression_question
)

# Optional Example — Wikipedia RAG without `WikipediaLoader`

The older `wikipedia` package can fail with `JSONDecodeError` when the API response is blocked or is not JSON.

This optional example calls the MediaWiki API directly using `requests`, supplies a User-Agent, converts the result into LangChain `Document` objects, and then uses the same RAG pipeline.

If Wikipedia is blocked by a corporate proxy/firewall, the cell prints a message and the rest of the notebook can still run.

In [ ]:
def load_wikipedia_documents(query, max_docs=2):
    api_url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": (
            "LangChain-RAG-Training-Demo/1.0 "
            "(educational notebook)"
        )
    }

    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "utf8": 1,
        "srlimit": max_docs
    }

    search_response = requests.get(
        api_url,
        params=search_params,
        headers=headers,
        timeout=30
    )
    search_response.raise_for_status()

    try:
        search_data = search_response.json()
    except ValueError as exc:
        preview = search_response.text[:250]
        raise RuntimeError(
            "Wikipedia did not return JSON. "
            f"Response preview: {preview!r}"
        ) from exc

    search_results = search_data.get(
        "query",
        {}
    ).get(
        "search",
        []
    )

    documents = []

    for result in search_results:
        title = result["title"]

        extract_params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": 1,
            "redirects": 1,
            "titles": title,
            "format": "json",
            "utf8": 1
        }

        extract_response = requests.get(
            api_url,
            params=extract_params,
            headers=headers,
            timeout=30
        )
        extract_response.raise_for_status()

        extract_data = extract_response.json()

        pages = extract_data.get(
            "query",
            {}
        ).get(
            "pages",
            {}
        )

        for page in pages.values():
            text = page.get("extract", "").strip()

            if text:
                documents.append(
                    Document(
                        page_content=text,
                        metadata={
                            "title": page.get(
                                "title",
                                title
                            ),
                            "source": (
                                "https://en.wikipedia.org/wiki/"
                                + title.replace(" ", "_")
                            )
                        }
                    )
                )

    return documents

In [ ]:
try:
    wiki_documents = load_wikipedia_documents(
        "MS Dhoni",
        max_docs=2
    )

    print(
        "Wikipedia documents loaded:",
        len(wiki_documents)
    )

except Exception as exc:
    wiki_documents = []

    print(
        "Wikipedia API is unavailable in this "
        "environment, so this optional example "
        "will be skipped."
    )
    print("Reason:", exc)

In [ ]:
if wiki_documents:
    wiki_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )

    wiki_chunks = wiki_splitter.split_documents(
        wiki_documents
    )

    WIKI_DB_DIR = "wiki_db_final"

    shutil.rmtree(
        WIKI_DB_DIR,
        ignore_errors=True
    )

    wiki_db = Chroma.from_documents(
        documents=wiki_chunks,
        embedding=embeddings,
        collection_name="wikipedia_demo",
        persist_directory=WIKI_DB_DIR
    )

    wiki_retriever = wiki_db.as_retriever(
        search_kwargs={"k": 4}
    )

    wiki_rag = create_rag_chain(
        wiki_retriever,
        llm
    )

    ask_rag(
        wiki_rag,
        "What is MS Dhoni's date of birth?"
    )
else:
    print(
        "Skipping Wikipedia vector DB and RAG "
        "because no Wikipedia documents were loaded."
    )

# Example 5 — Constitution of India RAG

This section keeps the Constitution-of-India use case, but uses the same modern RAG architecture and `ChatOpenAI`.

In [ ]:
COI_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/COI/COI.pdf"
)

coi_pdf = download_file(
    COI_URL,
    "COI.pdf"
)

coi_pages = PyPDFLoader(
    str(coi_pdf)
).load()

print("Pages loaded:", len(coi_pages))

In [ ]:
coi_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=300
)

coi_chunks = coi_splitter.split_documents(
    coi_pages
)

print("Chunks created:", len(coi_chunks))

In [ ]:
COI_DB_DIR = "coi_db_final"

shutil.rmtree(
    COI_DB_DIR,
    ignore_errors=True
)

coi_db = Chroma.from_documents(
    documents=coi_chunks,
    embedding=embeddings,
    collection_name="constitution_of_india",
    persist_directory=COI_DB_DIR
)

coi_retriever = coi_db.as_retriever(
    search_kwargs={"k": 5}
)

coi_rag = create_rag_chain(
    coi_retriever,
    llm
)

print("Constitution RAG chain created.")

In [ ]:
coi_response = ask_rag(
    coi_rag,
    (
        "According to the Constitution of India, "
        "what are the fundamental rights of citizens?"
    )
)

## Out-of-context test

A well-instructed RAG chain should avoid answering from general model knowledge when the retrieved context does not support the answer.

In [ ]:
ask_rag(
    coi_rag,
    "What is the current weather in Hyderabad?"
)

# Example 6 — Simple RAG validation using `ChatOpenAI`

Instead of the older `QAEvalChain`, this section uses a current runnable pipeline:

**Reference answer + RAG answer → Evaluation prompt → ChatOpenAI → PASS/FAIL + reason**

To keep API usage small, only the first few CSV rows are evaluated by default.

In [ ]:
COI_QA_URL = (
    "https://raw.githubusercontent.com/giridhar276/"
    "Datasets/master/COI/COI_Q_A.csv"
)

qa_csv = download_file(
    COI_QA_URL,
    "COI_Q_A.csv"
)

test_data = pd.read_csv(
    qa_csv
)

display(test_data.head())

In [ ]:
judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are evaluating a RAG answer.

Compare the generated answer with the reference answer.

Return exactly two lines:
Verdict: PASS or FAIL
Reason: one short reason

A PASS means the generated answer is substantively
consistent with the reference answer. Minor wording
differences are acceptable."""
        ),
        (
            "human",
            """Question:
{question}

Reference answer:
{reference_answer}

Generated RAG answer:
{generated_answer}"""
        )
    ]
)

judge_chain = (
    judge_prompt
    | llm
    | StrOutputParser()
)

print("RAG evaluation chain created.")

In [ ]:
MAX_EVAL_ROWS = min(
    5,
    len(test_data)
)

evaluation_rows = []

for _, row in test_data.head(MAX_EVAL_ROWS).iterrows():
    question = str(row["Question"])
    reference_answer = str(row["Answer"])

    rag_response = coi_rag.invoke(
        {"input": question}
    )

    generated_answer = rag_response[
        "answer"
    ]

    evaluation = judge_chain.invoke(
        {
            "question": question,
            "reference_answer": reference_answer,
            "generated_answer": generated_answer
        }
    )

    evaluation_rows.append(
        {
            "Question": question,
            "Reference Answer": reference_answer,
            "RAG Answer": generated_answer,
            "Evaluation": evaluation
        }
    )

evaluation_df = pd.DataFrame(
    evaluation_rows
)

display(evaluation_df)

In [ ]:
pass_count = (
    evaluation_df["Evaluation"]
    .str.contains(
        "Verdict: PASS",
        case=False,
        na=False
    )
    .sum()
)

total = len(evaluation_df)

print(f"PASS results: {pass_count}/{total}")

if total:
    print(
        "Approximate pass rate:",
        f"{(pass_count / total) * 100:.1f}%"
    )

# Final architecture recap

For every example in this notebook, the main RAG flow is:

```text
User Question
      |
      v
Retriever
      |
      v
Relevant Chunks from Chroma
      |
      v
Prompt + Retrieved Context
      |
      v
ChatOpenAI
      |
      v
Grounded Final Answer
```

### Key objects used

```python
OpenAIEmbeddings(...)
Chroma.from_documents(...)
vector_db.as_retriever(...)
retriever.invoke(...)
ChatOpenAI(...)
create_stuff_documents_chain(...)
create_retrieval_chain(...)
rag_chain.invoke({"input": question})
```

### MultiQuery version

```text
Question
   |
   v
ChatOpenAI generates alternative queries
   |
   v
MultiQueryRetriever
   |
   v
Combined unique chunks
   |
   v
ChatOpenAI generates final RAG answer
```

### Contextual compression version

```text
Question
   |
   v
Base Retriever
   |
   v
Candidate chunks
   |
   v
LLMChainExtractor
   |
   v
Compressed relevant context
   |
   v
ChatOpenAI generates final RAG answer
```